In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.common.exceptions import WebDriverException
from webdriver_manager.chrome import ChromeDriverManager
import concurrent.futures
import time

DEBUG_PORT = 9222

def try_connect_existing_chrome():
    options = webdriver.ChromeOptions()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_experimental_option(
        "debuggerAddress", f"127.0.0.1:{DEBUG_PORT}"
    )
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)


def get_or_create_driver(timeout=5):
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            print("🔁 Tentative de connexion à Chrome existant...")
            with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
                future = executor.submit(try_connect_existing_chrome)
                driver = future.result(timeout=timeout)
            print("✅ Connecté à Chrome existant")
            return driver
        except (WebDriverException, concurrent.futures.TimeoutError):
            print("⏳ Chrome non dispo ou timeout, retry...")
            time.sleep(0.5)

    # Après timeout → lancement d'un nouveau Chrome
    print("🚀 Timeout atteint → lancement d'un nouveau Chrome")
    options = webdriver.ChromeOptions()
    options.add_argument(f"--remote-debugging-port={DEBUG_PORT}")
    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    print("🆕 Nouveau Chrome lancé avec debugging")
    return driver


: 

In [2]:
driver = get_or_create_driver()
driver.get("https://dpm.lol/tierlist?tier=gold_plus")

🔁 Tentative de connexion à Chrome existant...
⏳ Chrome non dispo ou timeout, retry...
🚀 Timeout atteint → lancement d'un nouveau Chrome
🆕 Nouveau Chrome lancé avec debugging


In [3]:
def get_last_radix_buttons():
    """
    Récupère les boutons du dernier pop-up Radix ouvert.
    Utilise l'ID commençant par 'radix-' pour identifier le pop-up correct.
    """

    # 1️⃣ chercher tous les éléments dont l'ID commence par 'radix-'
    radix_roots = driver.find_elements(By.XPATH, "//*[starts-with(@id, 'radix-')]")
    if not radix_roots:

        return []

    # 2️⃣ prendre le dernier pop-up (le plus récemment ouvert)
    radix_root = radix_roots[-1]

    # 3️⃣ le div interne qui contient les boutons
    try:
        radix_div = radix_root.find_element(By.XPATH, "./div")
    except:

        return []

    # 4️⃣ récupérer les boutons
    buttons = radix_div.find_elements(By.TAG_NAME, "button")

    return buttons

In [ ]:
# def init_parse(driver, scroll_pause=1.0, scroll_step=500):

#     champions_by_key = {}

#     scroll_top = 0
#     print("🚀 init_parse() démarré")

#     while True:
#         driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_top)
#         time.sleep(scroll_pause)

#         container_selector = (
#             "#root > main > div > div.flex.justify-center.gap-16 > "
#             "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
#             "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
#             "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
#         )

#         container = driver.find_element(By.CSS_SELECTOR, container_selector)
#         rows = container.find_elements(By.XPATH, "./div/div")

#         for idx, row in enumerate(rows):
#             text = row.text.strip()
#             if not text:
#                 continue

#             lines = text.splitlines()
#             label = lines[0]
#             name = lines[1] if len(lines) > 1 else "Unknown"

#             key = text  # ou (label, name)

#             champions_by_key[key] = {
#                 "scroll_y": scroll_top,
#                 "label": label,
#                 "numero": idx,
#                 "name": name,
#             }

#         scroll_top += scroll_step
#         new_height = driver.execute_script("return document.body.scrollHeight")

#         if scroll_top >= new_height:
#             break

#     champions = list(champions_by_key.values())

#     print(f"✅ {len(champions)} champions collectés (dernières occurrences)")
#     return champions


In [4]:
def init_parse(driver, scroll_pause=1.0, scroll_step=500):

    champions_by_key = {}

    scroll_top = 0
    print("🚀 init_parse() démarré")

    while True:
        driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_top)
        time.sleep(scroll_pause)

        container_selector = (
            "#root > main > div > div.flex.justify-center.gap-16 > "
            "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
            "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
            "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
        )

        container = driver.find_element(By.CSS_SELECTOR, container_selector)
        rows = container.find_elements(By.XPATH, "./div/div")

        for idx, row in enumerate(rows):
            text = row.text.strip()
            if not text:
                continue

            lines = text.splitlines()
            label = lines[0]
            name = lines[1] if len(lines) > 1 else "Unknown"

            key = text  # ou (label, name)

            if key not in champions_by_key:
                champions_by_key[key] = {
                    "scroll_positions": [],
                    "label": label,
                    "numero": idx,
                    "name": name,
                }

            champions_by_key[key]["scroll_positions"].append(scroll_top)

        scroll_top += scroll_step
        new_height = driver.execute_script("return document.body.scrollHeight")

        if scroll_top >= new_height:
            break

    # 🔁 Construction finale : avant-dernier scroll si possible
    champions = []
    for champ in champions_by_key.values():
        positions = champ["scroll_positions"]
        champ["scroll_y"] = ((positions[-2] + positions[-1]) // 2 ) if len(positions) >= 2 else positions[-1]
        del champ["scroll_positions"]
        champions.append(champ)

    print(f"✅ {len(champions)} champions collectés (avant-dernière occurrence si dispo)")
    return champions


In [5]:
import hashlib
from selenium.webdriver.common.by import By

ROLE_HASH_TO_TEXT = {
    "df14d23b35c9842bd8afa0db2b922aaf": "top",
    "bd9e54c883010f9bc2487e7d26a91b77": "jun",
    "3ce111209b6d69bee8498e94b02567ad": "mid",
    "6f1d8859e29002c2c45b527544e5a755": "adc",
    "3b66426a9b2beeca218cc726985f68b1": "sup",
}

def resolve_role_from_hash(svg_hash: str) -> str:
    role = ROLE_HASH_TO_TEXT.get(svg_hash)

    if role is None:
        print(f"⚠️ Hash de rôle inconnu : {svg_hash}")
        return "unknown"

    return role

# svg_hash = hashlib.md5(role_svg.encode("utf-8")).hexdigest()

def hash_svg_path(svg_element) -> str:
    """
    Extrait le path 'd' du SVG et retourne son hash MD5
    """
    paths = svg_element.find_elements(By.TAG_NAME, "path")
    if not paths:
        return None

    path_d = paths[0].get_attribute("d").strip()
    return hashlib.md5(path_d.encode("utf-8")).hexdigest()


In [6]:
def get_selected_played_lane(driver) -> dict:
    print("🟢 Détection de la played lane sélectionnée")

    # 1️⃣ on récupère TOUS les containers possibles (sécurise si la page évolue)
    containers = driver.find_elements(
        By.CSS_SELECTOR,
        "div.border-black-600.flex.w-fit.items-center.justify-center"
    )

    print(f"🔍 {len(containers)} containers candidats trouvés")

    if not containers:
        print("❌ Aucun container trouvé")
        return {"lane": None, "percentage": None}

    # 2️⃣ on prend le premier (structure unique sur la page)
    container = containers[0]

    lane_divs = container.find_elements(By.XPATH, "./div")
    print(f"🔍 {len(lane_divs)} boutons de lane trouvés")

    # 3️⃣ on parcourt uniquement les 5 boutons
    for idx, lane_div in enumerate(lane_divs):
        try:
            link = lane_div.find_element(By.TAG_NAME, "a")
            inner_div = link.find_element(By.TAG_NAME, "div")

            class_name = inner_div.get_attribute("class") or ""
            print(f"[Lane {idx}] classes = {class_name}")

            # pas sélectionné → on skip
            if "bg-blue-200" not in class_name:
                continue

            print(f"⭐ Lane sélectionnée détectée (index {idx})")

            # SVG → hash → lane
            svg = inner_div.find_element(By.TAG_NAME, "svg")
            svg_hash = hash_svg_path(svg)
            lane = resolve_role_from_hash(svg_hash)

            print(f"   🔐 svg_hash = {svg_hash}")
            print(f"   🏷️ lane     = {lane}")

            # 4️⃣ span JUSTE APRÈS le <a>
            percentage = None
            try:
                span = lane_div.find_element(By.TAG_NAME, "span")
                percentage = span.text.strip()
            except Exception:
                print("⚠️ Span pourcentage introuvable")

            print(f"   📊 percentage = {percentage}")

            return {
                "lane": lane,
                "percentage": percentage
            }

        except Exception as e:
            print(f"[Lane {idx}] ❌ erreur : {e}")

    print("❌ Aucune lane sélectionnée trouvée")
    return {"lane": None, "percentage": None}


In [7]:
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException

def collect_champion_lane_stats(driver) -> dict:
    print("🟢 Collecte des stats champion / lane")

    stats = {
        "tier": None,
        "rank": None,
        "winrate": None,
        "pickrate": None,
        "banrate": None,
        "games": None,
    }

    try:
        container = driver.find_element(
            By.XPATH,
            '//*[@id="root"]/main/div/div[3]/div[2]/div/div[2]/div[1]'
        )
    except NoSuchElementException:
        print("❌ Container stats introuvable")
        return stats

    print("✅ Container stats trouvé")
    print("🎨 class :", container.get_attribute("class"))

    spans = container.find_elements(By.TAG_NAME, "span")
    print(f"🔢 {len(spans)} spans trouvés\n")

    KNOWN_KEYS = {
        "niveau": "tier",
        "rang": "rank",
        "winrate": "winrate",
        "pickrate": "pickrate",
        "banrate": "banrate",
        "parties": "games",
    }

    for idx, span in enumerate(spans):
        raw_text = span.text.strip()
        if not raw_text:
            continue

        print(f"[Span {idx}] → '{raw_text}'")

        lower = raw_text.lower()

        for label, key in KNOWN_KEYS.items():
            if label in lower:
                # ne pas écraser une valeur déjà trouvée
                if stats[key] is not None:
                    continue

                # extraction valeur
                value = (
                    raw_text
                    .replace(label, "")
                    .replace("\n", " ")
                    .strip()
                )

                # ignorer les spans "label only"
                if not value:
                    continue

                # suppression de l'espace insécable pour "games"
                if key == "games":
                    value = value.replace("\u202f", "").replace(" ", "")

                stats[key] = value
                print(f"✅ {key} détecté → '{value}'")

    print("-" * 60)
    print("✅ Stats collectées :", stats)
    return stats


In [8]:
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, WebDriverException
import time

def click_matchup_or_synergy(driver, matchup: bool) -> bool:
    target_name = "MATCHUPS" if matchup else "SYNERGIES"
    aria_key = "Enemy" if matchup else "Ally"

    print(f"🟢 Sélection de l'onglet {target_name}", flush=True)

    xpath = f"//button[@role='tab' and contains(@aria-controls, '{aria_key}')]"

    try:
        button = driver.find_element(By.XPATH, xpath)

        data_state = button.get_attribute("data-state")
        print(f"🔍 {target_name} data-state = {data_state}")

        if data_state == "active":
            print(f"ℹ️ Onglet {target_name} déjà actif")
            return False

        driver.execute_script(
            "arguments[0].scrollIntoView({block: 'center'});",
            button
        )
        time.sleep(0.3)

        button.click()
        print(f"🖱️ Onglet {target_name} cliqué", flush=True)
        time.sleep(0.7)

        return True

    except NoSuchElementException:
        print(f"❌ Onglet {target_name} introuvable (aria-controls)", flush=True)
        return False

    except WebDriverException as e:
        print(f"❌ Erreur Selenium sur {target_name} : {e}", flush=True)
        return False


In [9]:
def click_matchup_or_synergy_lane(driver, lane_name) -> bool:
    print(f"🟢 Sélection de la lane {lane_name} via hash du PATH SVG")

    time.sleep(0.5)

    lane_container_xpath = (
        '//*[@id="root"]/main[1]/div[1]/div[3]/div[2]/div[1]/div[2]/div[4]'
        '/div[1]/div[1]/div[3]/div[1]'
    )

    try:
        lane_container = driver.find_element(By.XPATH, lane_container_xpath)
    except Exception as e:
        print(f"❌ Conteneur des lanes introuvable : {e}")
        return False

    lane_buttons = lane_container.find_elements(By.TAG_NAME, "button")
    print(f"🔍 {len(lane_buttons)} boutons de lane trouvés\n")

    for i, btn in enumerate(lane_buttons):
        svgs = btn.find_elements(By.TAG_NAME, "svg")
        if not svgs:
            continue

        svg = svgs[0]
        svg_hash = hash_svg_path(svg)

        if not svg_hash:
            continue

        role = resolve_role_from_hash(svg_hash)

        print(f"[Lane {i}] hash={svg_hash} role={role}")

        if role == lane_name:
            print(f"🎯 Bouton {lane_name} identifié — clic")

            driver.execute_script(
                "arguments[0].scrollIntoView({block: 'center'});",
                btn,
            )
            time.sleep(0.3)
            btn.click()

            print(f"🖱️ Bouton {lane_name} cliqué avec succès")
            return True

    print(f"❌ Bouton {lane_name} non trouvé")
    return False


In [10]:
from selenium.webdriver.common.by import By
from selenium.common.exceptions import (
    NoSuchElementException,
    StaleElementReferenceException,
    ElementClickInterceptedException,
    ElementNotInteractableException,
)
import time


def click_liste_complete(driver) -> bool:
    print("🟢 Recherche du bouton '+ Liste complète' parmi tous les boutons...", flush=True)

    for attempt in range(2):
        print(f"🔁 Tentative {attempt + 1}/2")

        try:
            buttons = driver.find_elements(By.TAG_NAME, "button")
            print(f"🔍 {len(buttons)} boutons trouvés sur la page")

            for i, btn in enumerate(buttons):
                try:
                    btn_text = btn.text.strip()
                    btn_text_clean = " ".join(btn_text.split())

                    # XPath absolu (debug)
                    btn_xpath = driver.execute_script(
                        """
                        function getXPath(element) {
                            if (element.id !== '')
                                return '//*[@id="' + element.id + '"]';
                            if (element === document.body)
                                return '/html/body';

                            let ix = 0;
                            let siblings = element.parentNode.childNodes;
                            for (let i = 0; i < siblings.length; i++) {
                                let sibling = siblings[i];
                                if (sibling === element)
                                    return getXPath(element.parentNode) + '/' +
                                        element.tagName.toLowerCase() + '[' + (ix + 1) + ']';
                                if (sibling.nodeType === 1 && sibling.tagName === element.tagName)
                                    ix++;
                            }
                        }
                        return getXPath(arguments[0]);
                        """,
                        btn,
                    )

                    print(f"[{i}] → '{btn_text_clean}' | XPath: {btn_xpath}")

                    if "+ Liste complète" in btn_text_clean:
                        print("🎯 Bouton '+ Liste complète' détecté, tentative de clic...")

                        driver.execute_script(
                            "arguments[0].scrollIntoView({block: 'center'});",
                            btn,
                        )
                        time.sleep(0.75)
                        btn.click()

                        print("🖱️ Bouton '+ Liste complète' cliqué avec succès", flush=True)
                        time.sleep(1)
                        return True  # ✅ succès immédiat

                except StaleElementReferenceException:
                    print(f"⚠️ Bouton [{i}] devenu obsolète (DOM mis à jour)")
                except Exception as e:
                    print(f"⚠️ Erreur sur le bouton [{i}] : {e}")

        except (
            NoSuchElementException,
            ElementClickInterceptedException,
            ElementNotInteractableException,
        ) as e:
            print(f"❌ Erreur lors de la tentative {attempt + 1} : {e}")

        time.sleep(0.5)

    print("❌ Bouton '+ Liste complète' non cliqué après 2 tentatives", flush=True)
    return False  # ❌ échec final


In [ ]:
# from selenium.webdriver.common.by import By
# from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException
# import time

# def collect_matchup(driver):
#     """
#     Parcourt les matchups Enemy après clic sur 'Liste complète'
#     et retourne une liste de dicts :
#     {
#         champ_counter_i,
#         winrate,
#         games,
#         lane_quality
#     }
#     """

#     print("🟢 Démarrage collect_matchup")
#     results = []
#     seen = set()

#     # ===============================
#     # 1️⃣ Section Enemy
#     # ===============================
#     # try:
#     #     enemy_root = driver.find_element(By.XPATH, '//*[@id="radix-_r_8_-content-Enemy"]')
#     #     print(f"✅ Section Enemy trouvée (ID: {enemy_root.get_attribute('id')})")
#     # except NoSuchElementException:
#     #     print("❌ Section Enemy introuvable !")
#     #     return results

#     enemy_root = None
#     for attempt in range(2):
#         try:
#             enemy_root = driver.find_element(By.XPATH, '//*[@id="radix-_r_8_-content-Enemy"]')
#             print(f"✅ Section Enemy trouvée (ID: {enemy_root.get_attribute('id')})")
#             break
#         except NoSuchElementException:
#             print(f"⏳ Tentative {attempt + 1}/2 : Section Enemy introuvable")
#             time.sleep(0.8)  # petit sleep (ajuste si besoin)

#     if enemy_root is None:
#         print("❌ Section Enemy introuvable après 2 tentatives !")
#         return results

#     # ===============================
#     # 2️⃣ Récupérer les deux wrappers : visible + hors écran
#     # ===============================
#     try:
#         visible_wrapper = enemy_root.find_element(By.XPATH, "./div/div[1]/div")
#         hidden_wrapper = enemy_root.find_element(By.XPATH, "./div/div[2]/div")
#         print("✅ Wrappers visible et hidden trouvés")
#     except NoSuchElementException:
#         print("❌ Wrappers introuvables !")
#         return results

#     # ===============================
#     # 3️⃣ Fonction de parsing des cards
#     # ===============================
#     def parse_cards(container):
#         cards = container.find_elements(By.XPATH, "./div")
#         print(f"🔍 {len(cards)} cards trouvées dans ce container")
#         for idx, card in enumerate(cards):
#             try:
#                 # ---------- <a> ----------
#                 anchor = card.find_element(By.XPATH, "./a")
#                 img = anchor.find_element(By.XPATH, ".//img")
#                 champ_name = img.get_attribute("alt").strip()
#                 if not champ_name or champ_name in seen:
#                     continue
#                 seen.add(champ_name)

#                 # -------- winrate et nombre de parties --------
#                 info_div = anchor.find_element(By.XPATH, "./div/div[2]")  # div C
#                 spans = info_div.find_elements(By.XPATH, "./span")
#                 winrate = spans[0].text.strip() if len(spans) > 0 else ""
#                 games = spans[1].text.strip() if len(spans) > 1 else ""

#                 # -------- lane_quality --------
#                 lane_quality = ""
#                 try:
#                     button = card.find_element(By.XPATH, "./button")
#                     lane_quality = button.text.strip()
#                 except NoSuchElementException:
#                     pass

#                 print(f"🎯 {champ_name} | Winrate: {winrate} | Games: {games} | Lane: {lane_quality}")

#                 results.append({
#                     "champ_counter_i": champ_name,
#                     "winrate": winrate,
#                     "games": games,
#                     "lane_quality": lane_quality
#                 })

#             except StaleElementReferenceException:
#                 print("⚠️ StaleElement — skip")
#             except Exception as e:
#                 print(f"❌ Erreur card {idx} → {e}")

#     # ===============================
#     # 4️⃣ Parser les cards visibles et hors écran
#     # ===============================
#     parse_cards(visible_wrapper)
#     parse_cards(hidden_wrapper)

#     print(f"📦 Matchups collectés : {len(results)}")
#     return results


In [11]:
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException
import time

def collect_matchup(driver, matchup: bool = True):
    """
    Parcourt les matchups (Enemy) ou synergies (Ally) après clic sur 'Liste complète'
    et retourne une liste de dicts :
    {
        champ_counter_i,
        winrate,
        games,
        lane_quality
    }

    :param matchup: True pour matchups (Enemy), False pour synergies (Ally)
    """

    print("🟢 Démarrage collect_matchup")
    results = []
    seen = set()

    # ===============================
    # 1️⃣ Section Enemy / Ally
    # ===============================
    aria_key = "Enemy" if matchup else "Ally"
    container = None
    for attempt in range(2):
        try:
            # on sélectionne le container via role="tabpanel" et aria-labelledby
            container = driver.find_element(
                By.XPATH,
                f"//div[@role='tabpanel' and contains(@aria-labelledby, '{aria_key}')]"
            )
            print(f"✅ Section {'Enemy' if matchup else 'Ally'} trouvée (aria-labelledby: {aria_key})")
            break
        except NoSuchElementException:
            print(f"⏳ Tentative {attempt+1}/2 : Section {'Enemy' if matchup else 'Ally'} introuvable")
            time.sleep(0.8)

    if container is None:
        print(f"❌ Section {'Enemy' if matchup else 'Ally'} introuvable après 2 tentatives !")
        return results

    # ===============================
    # 2️⃣ Récupérer les deux wrappers : visible + hors écran
    # ===============================
    try:
        visible_wrapper = container.find_element(By.XPATH, "./div/div[1]/div")
        hidden_wrapper = container.find_element(By.XPATH, "./div/div[2]/div")
        print("✅ Wrappers visible et hidden trouvés")
    except NoSuchElementException:
        print("❌ Wrappers introuvables !")
        return results

    # ===============================
    # 3️⃣ Fonction de parsing des cards
    # ===============================
    def parse_cards(wrapper):
        cards = wrapper.find_elements(By.XPATH, "./div")
        print(f"🔍 {len(cards)} cards trouvées dans ce wrapper")
        for idx, card in enumerate(cards):
            try:
                # ---------- <a> ----------
                anchor = card.find_element(By.XPATH, "./a")
                img = anchor.find_element(By.XPATH, ".//img")
                champ_name = img.get_attribute("alt").strip()
                if not champ_name or champ_name in seen:
                    continue
                seen.add(champ_name)

                # -------- winrate et nombre de parties --------
                info_div = anchor.find_element(By.XPATH, "./div/div[2]")  # div C
                spans = info_div.find_elements(By.XPATH, "./span")
                winrate = spans[0].text.strip() if len(spans) > 0 else ""
                games = spans[1].text.strip().replace("\u202f", "") if len(spans) > 1 else ""

                # -------- lane_quality --------
                lane_quality = ""
                try:
                    button = card.find_element(By.XPATH, "./button")
                    lane_quality = button.text.strip()
                except NoSuchElementException:
                    pass

                print(f"🎯 {champ_name} | Winrate: {winrate} | Games: {games} | Lane: {lane_quality}")

                results.append({
                    "champ_counter_i": champ_name,
                    "winrate": winrate,
                    "games": games,
                    "lane_quality": lane_quality
                })

            except StaleElementReferenceException:
                print("⚠️ StaleElement — skip")
            except Exception as e:
                print(f"❌ Erreur card {idx} → {e}")

    # ===============================
    # 4️⃣ Parser les cards visibles et hors écran
    # ===============================
    parse_cards(visible_wrapper)
    parse_cards(hidden_wrapper)

    print(f"📦 {'Matchups' if matchup else 'Synergies'} collectés : {len(results)}")
    return results


In [ ]:
# from selenium.webdriver.common.by import By
# from selenium.webdriver.common.action_chains import ActionChains
# from selenium.webdriver.common.keys import Keys
# from selenium.common.exceptions import StaleElementReferenceException
# import time


# def open_champion_in_new_tab(
#     driver,
#     champion_descriptor: dict,
#     wait_seconds: int = 5,
# ):
#     """
#     champion_descriptor = {
#         "global_index": int,
#         "scroll_y": int,
#         "label": str
#     }
#     """

#     container_selector = (
#         "#root > main > div > div.flex.justify-center.gap-16 > "
#         "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
#         "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
#         "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
#     )

#     scroll_y = champion_descriptor["scroll_y"]
#     label = champion_descriptor["label"]
#     numero= champion_descriptor["numero"]

#     print(f"🧭 Scroll vers Y={scroll_y} pour {label}")
#     driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_y)
#     time.sleep(0.5)  # 👈 important

#     container = driver.find_element(By.CSS_SELECTOR, container_selector)
#     rows = container.find_elements(By.XPATH, "./div/div")

#     if not rows:
#         print("❌ Aucun champion visible après scroll")
#         return

#     row = rows[numero]  # premier visible à cet endroit
#     print(f"🖱️ Ouverture champion → {label}")

#     main_window = driver.current_window_handle

#     ActionChains(driver) \
#         .key_down(Keys.CONTROL) \
#         .click(row) \
#         .key_up(Keys.CONTROL) \
#         .perform()

#     time.sleep(2)

#     windows = driver.window_handles
#     if len(windows) < 2:
#         print("❌ Nouvel onglet non détecté")
#         return

#     new_tab = [w for w in windows if w != main_window][0]

#     for i in range(1, wait_seconds - 1):
#         print(f"⏳ {i}/{wait_seconds}s")
#         time.sleep(1)
#     driver.switch_to.window(new_tab)
#     for i in range(1, wait_seconds):
#         print(f"⏳ {i}/{wait_seconds}s")
#         time.sleep(1)
#     for i in range(1, wait_seconds):
#         print(f"⏳ {i}/{wait_seconds}s")
#         time.sleep(1)


#     # dans l'ordre : 
#     # initialiser data_champ 
#     data_champ = { 
#     "name": champion_descriptor["name"],
#     }
#     # Ajouter ces lignes avant d'itérer sur les rôles
#     data_champ["matchups"] = {}   # ✅ initialisation pour éviter KeyError
#     data_champ["synergies"] = {}  # ✅ initialisation pour éviter KeyError
#     # collecter les données du champion dont role_champ, mettre ces données dans le dictionnaire data_champ
#     lane_and_percentage = get_selected_played_lane(driver)
#     # {"lane": None, "percentage": None}
#     data_champ["role"] = lane_and_percentage["lane"]
#     data_champ["role_play_ratio"] = lane_and_percentage["percentage"]
#     general_data_for_champ_at_lane = collect_champion_lane_stats(driver)
#     data_champ["tier"] = general_data_for_champ_at_lane["tier"]
#     data_champ["rank"] = general_data_for_champ_at_lane["rank"]
#     data_champ["winrate"] = general_data_for_champ_at_lane["winrate"]
#     data_champ["pickrate"] = general_data_for_champ_at_lane["pickrate"]
#     data_champ["banrate"] = general_data_for_champ_at_lane["banrate"]
#     data_champ["nb_games_analyzed"] = general_data_for_champ_at_lane["games"]
#     print(data_champ)
#     # cliquer sur matchups
#     click_matchup_or_synergy(driver, matchup=True)
#     roles = ["top", "jun", "mid", "adc", "sup"]
#     # # itérer sur les roles
#     # for role in roles:
#     #     print(role)
#     #     # stocker le role du vis à vis
#     #     # cliquer sur le bouton du role
#     #     click_matchup_or_synergy_lane(driver, role)
#     #     time.sleep(1.5)
#     #     # click_liste_complete (que la premiere fois ?) en stockant dans matchups complet ()
#     #     complet = click_liste_complete(driver)
#     #     print("complet ? ", complet)
#     #     time.sleep(2)
#     #     # collect_matchup -> stocker dans matchup_role
#     #     matchup = collect_matchup(driver, matchup=True)
#     #     # ajouter matchup_role dans data_champ["matchups"][role_champ] = matchup_role
#     #     data_champ["matchups"][role] = matchup
#     # # cliquer sur synergy
#     # click_matchup_or_synergy(driver, matchup=False)
#     # synergy_roles = [r for r in roles if r != data_champ["role"]]
#     # # itérer sur les roles - role_champ
#     # for role in synergy_roles:
#     #     print(f"Collecte synergy avec le rôle : {role}")
#     #     # stocker le role du vis à vis
#     #     # cliquer sur le bouton du role
#     #     click_matchup_or_synergy_lane(driver, role)
#     #     time.sleep(1.5)
#     #     # click_liste_complete (que la premiere fois ?) en stockant dans synergies complet ()
#     #     complet = click_liste_complete(driver)
#     #     print("complet ? ", complet)
#     #     time.sleep(2)
#     #     # collect_matchup -> stocker dans synergy_role
#     #     synergy = collect_matchup(driver, matchup=False)
#     #     # ajouter synergy_role dans data_champ["synergies"][role_champ] = synergy_role
#     #     data_champ["synergies"][role] = synergy

#     first_time_matchups = True
#     for role in roles:
#         print(f"Collecte matchup vs rôle : {role}")
#         # cliquer sur le bouton du rôle
#         click_matchup_or_synergy_lane(driver, role)
#         time.sleep(1.5)
#         # cliquer sur "liste complète" UNE SEULE FOIS
#         if first_time_matchups:
#             complet = click_liste_complete(driver)
#             print("matchups complets ?", complet)
#             first_time_matchups = False
#             time.sleep(2)
#         # collecter les matchups
#         matchup = collect_matchup(driver, matchup=True)
#         # stocker
#         data_champ["matchups"][role] = matchup
#     # --- SYNERGIES ---
#     click_matchup_or_synergy(driver, matchup=False)

#     synergy_roles = [r for r in roles if r != data_champ["role"]]
#     first_time_synergies = True
#     for role in synergy_roles:
#         print(f"Collecte synergy avec le rôle : {role}")
#         # cliquer sur le bouton du rôle
#         click_matchup_or_synergy_lane(driver, role)
#         time.sleep(1.5)
#         # cliquer sur "liste complète" UNE SEULE FOIS
#         if first_time_synergies:
#             complet = click_liste_complete(driver)
#             print("synergies complètes ?", complet)
#             first_time_synergies = False
#             time.sleep(2)
#         # collecter les synergies
#         synergy = collect_matchup(driver, matchup=False)
#         # stocker
#         data_champ["synergies"][role] = synergy

#     print(data_champ)

#     for i in range(1, wait_seconds + 3):
#         print(f"⏳ {i}/{wait_seconds}s")
#         time.sleep(1)
#     driver.close()
#     time.sleep(1)
#     driver.switch_to.window(main_window)
#     print("↩️ Retour liste champions")

#     return data_champ


In [12]:
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import StaleElementReferenceException
import time


def open_champion_in_new_tab(
    driver,
    champion_descriptor: dict,
    wait_seconds: int = 5,
):
    """
    champion_descriptor = {
        "global_index": int,
        "scroll_y": int,
        "label": str
    }
    """

    container_selector = (
        "#root > main > div > div.flex.justify-center.gap-16 > "
        "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
        "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
        "div.flex.flex-col.gap-y-8.w-full.mb-48 > div:nth-child(3) > div"
    )

    scroll_y = champion_descriptor["scroll_y"]
    label = champion_descriptor["label"]
    numero= champion_descriptor["numero"]

    print(f"🧭 Scroll vers Y={scroll_y} pour {label}")
    driver.execute_script("window.scrollTo(0, arguments[0]);", scroll_y)
    time.sleep(0.5)  # 👈 important

    container = driver.find_element(By.CSS_SELECTOR, container_selector)
    rows = container.find_elements(By.XPATH, "./div/div")

    if not rows:
        print("❌ Aucun champion visible après scroll")
        return

    row = rows[numero]  # premier visible à cet endroit
    print(f"🖱️ Ouverture champion → {label}")

    main_window = driver.current_window_handle

    ActionChains(driver) \
        .key_down(Keys.CONTROL) \
        .click(row) \
        .key_up(Keys.CONTROL) \
        .perform()

    time.sleep(2)

    windows = driver.window_handles
    if len(windows) < 2:
        print("❌ Nouvel onglet non détecté")
        return

    new_tab = [w for w in windows if w != main_window][0]

    for i in range(1, wait_seconds - 1):
        print(f"⏳ {i}/{wait_seconds}s")
        time.sleep(1)
    driver.switch_to.window(new_tab)
    for i in range(1, wait_seconds):
        print(f"⏳ {i}/{wait_seconds}s")
        time.sleep(1)
    for i in range(1, wait_seconds):
        print(f"⏳ {i}/{wait_seconds}s")
        time.sleep(1)


    # dans l'ordre : 
    # initialiser data_champ 
    data_champ = { 
    "name": champion_descriptor["name"],
    }
    # Ajouter ces lignes avant d'itérer sur les rôles
    data_champ["matchups"] = {}   # ✅ initialisation pour éviter KeyError
    data_champ["synergies"] = {}  # ✅ initialisation pour éviter KeyError
    # collecter les données du champion dont role_champ, mettre ces données dans le dictionnaire data_champ
    lane_and_percentage = get_selected_played_lane(driver)
    # {"lane": None, "percentage": None}
    data_champ["role"] = lane_and_percentage["lane"]
    data_champ["role_play_ratio"] = lane_and_percentage["percentage"]
    general_data_for_champ_at_lane = collect_champion_lane_stats(driver)
    data_champ["tier"] = general_data_for_champ_at_lane["tier"]
    data_champ["rank"] = general_data_for_champ_at_lane["rank"]
    data_champ["winrate"] = general_data_for_champ_at_lane["winrate"]
    data_champ["pickrate"] = general_data_for_champ_at_lane["pickrate"]
    data_champ["banrate"] = general_data_for_champ_at_lane["banrate"]
    data_champ["nb_games_analyzed"] = general_data_for_champ_at_lane["games"]
    print(data_champ)
    # cliquer sur matchups
    click_matchup_or_synergy(driver, matchup=True)
    roles = ["top", "jun", "mid", "adc", "sup"]
    # roles = ["top"]

    
    first_time_matchups = True
    for role in roles:
        print(f"Collecte matchup vs rôle : {role}")
        # cliquer sur le bouton du rôle
        click_matchup_or_synergy_lane(driver, role)
        time.sleep(1.5)
        # cliquer sur "liste complète" UNE SEULE FOIS
        if first_time_matchups:
            complet = click_liste_complete(driver)
            print("matchups complets ?", complet)
            first_time_matchups = False
            time.sleep(2)
        # collecter les matchups
        matchup = collect_matchup(driver, matchup=True)
        # stocker
        data_champ["matchups"][role] = matchup
    # --- SYNERGIES ---
    click_matchup_or_synergy(driver, matchup=False)

    synergy_roles = [r for r in roles if r != data_champ["role"]]
    first_time_synergies = True
    for role in synergy_roles:
        print(f"Collecte synergy avec le rôle : {role}")
        # cliquer sur le bouton du rôle
        click_matchup_or_synergy_lane(driver, role)
        time.sleep(1.5)
        # cliquer sur "liste complète" UNE SEULE FOIS
        if first_time_synergies:
            complet = click_liste_complete(driver)
            print("synergies complètes ?", complet)
            first_time_synergies = False
            time.sleep(2)
        # collecter les synergies
        synergy = collect_matchup(driver, matchup=False)
        # stocker
        data_champ["synergies"][role] = synergy

    print(data_champ)

    for i in range(1, wait_seconds + 3):
        print(f"⏳ {i}/{wait_seconds}s")
        time.sleep(1)
    driver.close()
    time.sleep(1)
    driver.switch_to.window(main_window)
    print("↩️ Retour liste champions")

    return data_champ


In [47]:
driver = get_or_create_driver()
driver.get("https://dpm.lol/tierlist?tier=gold_plus")

🔁 Tentative de connexion à Chrome existant...
✅ Connecté à Chrome existant


In [ ]:
# import pandas as pd

# def generate_csv_from_champions(all_champions_data, elo, server, patch, output_path="matchups_champions.csv"):
#     """
#     all_champions_data = [
#         {
#             "label": "Ahri",
#             "matchups": [
#                 {"champ_counter_i": "LeBlanc", "winrate": "52%", "games": "120", "lane_quality": "Bad Lane"},
#                 ...
#             ]
#         },
#         ...
#     ]
#     """
#     # 1️⃣ Déterminer le nombre max de matchups
#     max_matchups = max(len(champ["matchups"]) for champ in all_champions_data)

#     rows = []
#     print(f"🔍 Génération CSV avec {len(all_champions_data)} champions et max {max_matchups} matchups chacun")

#     for champ in all_champions_data:
#         print(champ)
#         row = {
#             "champion": champ["name"],
#             "role": None,
#             "ratio_games_this_role": None,
#             "tier": None,
#             "rank": None,
#             "winrate": None,
#             "pickrate": None,
#             "banrate": None,
#             "games_played": None,
#             "matchups_complets": champ.get("matchups_complets", None),
#         }

#         # Ajouter les colonnes pour chaque matchup
#         for i in range(max_matchups):
#             #selon si on est en synergie ou matchup, on nomme le prefixe differement!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
#             matchup_prefix = f"matchup_{i+1}"
#             if i < len(champ["matchups"]):
#                 m = champ["matchups"][i]
#                 row[f"{matchup_prefix}_name"] = m["champ_counter_i"]
#                 row[f"{matchup_prefix}_winrate"] = m["winrate"]
#                 row[f"{matchup_prefix}_parties_observees"] = m["games"]
#                 row[f"{matchup_prefix}_qualite_lane"] = m["lane_quality"]
#                 row[f"{matchup_prefix}_role"] = None  # pour l'instant
#             else:
#                 # Pas assez de matchups → remplir avec None
#                 row[f"{matchup_prefix}_name"] = None
#                 row[f"{matchup_prefix}_winrate"] = None
#                 row[f"{matchup_prefix}_parties_observees"] = None
#                 row[f"{matchup_prefix}_qualite_lane"] = None
#                 row[f"{matchup_prefix}_role"] = None

#         rows.append(row)

#     df = pd.DataFrame(rows)

#     # Nom du fichier avec le format que tu veux
#     filename = f"{output_path.replace('.csv','')}_{elo}_{server}_{patch}.csv"

#     df.to_csv(filename, index=False)
#     print(f"✅ CSV généré : {filename}")






# je veux que tu me réécrives la méthode suivantes pour qu'elle traite la nouvelle structure de donnée à la place de l'ancienne et qu'elle l'écrive sous forme de DF puis qu'elle enregistre ce DF dans un CSV. - la structure de doonnée présente 1 champion. je veux que mon DF ait une ligne par champion traité, avec tous ses matcuhps et synergies. Je veux que la fonction que tu me crées ait la meme signature et fasse la meme chose avec les parametres (sauf avec all_champions_data bienentendu puisque la structure change). je veux que pour chaque ligne du DF que tu me crées, les premieres colonnes soient x: "18.4 %" "banrate": str, # ex: "33.0 %" "nb_games_analyzed": str, # ex: "118" et ensuite, pour chaque colonne matchup (ou synergie), tu dois prefixer le nom de la colonne comme suit : matchup_[lane du matchup]_[numéro du matchup]. (lane du matchup n'est pas la lane du champion analysé, mais bel et bien la lane du matchup que tu trouves ici dans la structure de donnée : "matchups": { "top": [ { donc on va avoir 5 "types de matchups" : matchup_top... matchup_sup et 5 types de synergies. je veux que chaque type de matchup soit présent dans le DF autant de fois que le max de ce type de matchup apparait dans nos données. (tu completes avec des None biensur) donc voici encore une fois la structure du type de données : { "name": str, # ex: "Aphelios" "role": str, # ex: "adc" "role_play_ratio": str, # ex: "99.6%" "tier": str, # ex: "S+" "rank": str, # ex: "1 / 2" "winrate": str, # ex: "55.9 %" "pickrate": str, # ex: "18.4 %" "banrate": str, # ex: "33.0 %" "nb_games_analyzed": str, # ex: "118" "matchups": { "top": [ { "champ_counter_i": str, "winrate": str, "games": str, "lane_quality": str }, ... ], "jun": [ {...}, ... ], "mid": [ {...}, ... ], "adc": [ {...}, ... ], "sup": [ {...}, ... ] }, "synergies": { "top": [ { "champ_counter_i": str, "winrate": str, "games": str, "lane_quality": str }, ... ], "jun": [ {...}, ... ], "mid": [ {...}, ... ], "adc": [ {...}, ... ], "sup": [ {...}, ... ] } } et voici la methode que tu dois réécrire : import pandas as pd def generate_csv_from_champions(all_champions_data, elo, server, patch, output_path="matchups_champions.csv"): """ all_champions_data = [ { "label": "Ahri", "matchups": [ {"champ_counter_i": "LeBlanc", "winrate": "52%", "games": "120", "lane_quality": "Bad Lane"}, ... ] }, ... ] """ # 1️⃣ Déterminer le nombre max de matchups max_matchups = max(len(champ["matchups"]) for champ in all_champions_data) rows = [] print(f"🔍 Génération CSV avec {len(all_champions_data)} champions et max {max_matchups} matchups chacun") for champ in all_champions_data: print(champ) row = { "champion": champ["name"], "role": None, "ratio_games_this_role": None, "tier": None, "rank": None, "winrate": None, "pickrate": None, "banrate": None, "games_played": None, "matchups_complets": champ.get("matchups_complets", None), } # Ajouter les colonnes pour chaque matchup for i in range(max_matchups): #selon si on est en synergie ou matchup, on nomme le prefixe differement!!!!!!!!!!!!!!!!!!!!!!!!!!!!!! matchup_prefix = f"matchup_{i+1}" if i < len(champ["matchups"]): m = champ["matchups"][i] row[f"{matchup_prefix}_name"] = m["champ_counter_i"] row[f"{matchup_prefix}_winrate"] = m["winrate"] row[f"{matchup_prefix}_parties_observees"] = m["games"] row[f"{matchup_prefix}_qualite_lane"] = m["lane_quality"] row[f"{matchup_prefix}_role"] = None # pour l'instant else: # Pas assez de matchups → remplir avec None row[f"{matchup_prefix}_name"] = None row[f"{matchup_prefix}_winrate"] = None row[f"{matchup_prefix}_parties_observees"] = None row[f"{matchup_prefix}_qualite_lane"] = None row[f"{matchup_prefix}_role"] = None rows.append(row) df = pd.DataFrame(rows) # Nom du fichier avec le format que tu veux filename = f"{output_path.replace('.csv','')}_{elo}_{server}_{patch}.csv" df.to_csv(filename, index=False) print(f"✅ CSV généré : {filename}")


In [13]:
import pandas as pd


def generate_csv_from_champions(
    all_champions_data,
    elo,
    server,
    patch,
    output_path="matchups_champions.csv",
):
    """
    all_champions_data = [
        {
            "name": str,
            "role": str,
            "role_play_ratio": str,
            "tier": str,
            "rank": str,
            "winrate": str,
            "pickrate": str,
            "banrate": str,
            "nb_games_analyzed": str,

            "matchups": {
                "top": [ {...}, ... ],
                "jun": [ {...}, ... ],
                "mid": [ {...}, ... ],
                "adc": [ {...}, ... ],
                "sup": [ {...}, ... ],
            },

            "synergies": {
                "top": [ {...}, ... ],
                "jun": [ {...}, ... ],
                "mid": [ {...}, ... ],
                "adc": [ {...}, ... ],
                "sup": [ {...}, ... ],
            }
        },
        ...
    ]
    """

    lanes = ["top", "jun", "mid", "adc", "sup"]

    # ==========================================================
    # 1️⃣ Calculer les max par TYPE (matchup/synergy) et par LANE
    # ==========================================================
    max_matchups = {lane: 0 for lane in lanes}
    max_synergies = {lane: 0 for lane in lanes}

    for champ in all_champions_data:
        for lane in lanes:
            max_matchups[lane] = max(
                max_matchups[lane],
                len(champ.get("matchups", {}).get(lane, []))
            )
            max_synergies[lane] = max(
                max_synergies[lane],
                len(champ.get("synergies", {}).get(lane, []))
            )

    print("📊 Max matchups par lane :", max_matchups)
    print("📊 Max synergies par lane :", max_synergies)

    # ======================
    # 2️⃣ Construire les rows
    # ======================
    rows = []

    for champ in all_champions_data:
        row = {
            # -------- Colonnes principales (EN PREMIER) --------
            "champion": champ.get("name"),
            "role": champ.get("role"),
            "role_play_ratio": champ.get("role_play_ratio"),
            "tier": champ.get("tier"),
            "rank": champ.get("rank"),
            "winrate": champ.get("winrate"),
            "pickrate": champ.get("pickrate"),
            "banrate": champ.get("banrate"),
            "nb_games_analyzed": champ.get("nb_games_analyzed"),
        }

        # ======================
        # 3️⃣ MATCHUPS
        # ======================
        for lane in lanes:
            lane_matchups = champ.get("matchups", {}).get(lane, [])

            for i in range(max_matchups[lane]):
                prefix = f"matchup_{lane}_{i+1}"

                if i < len(lane_matchups):
                    m = lane_matchups[i]
                    row[f"{prefix}_name"] = m.get("champ_counter_i")
                    row[f"{prefix}_winrate"] = m.get("winrate")
                    row[f"{prefix}_games"] = m.get("games")
                    row[f"{prefix}_lane_quality"] = m.get("lane_quality")
                else:
                    row[f"{prefix}_name"] = None
                    row[f"{prefix}_winrate"] = None
                    row[f"{prefix}_games"] = None
                    row[f"{prefix}_lane_quality"] = None

        # ======================
        # 4️⃣ SYNERGIES
        # ======================
        for lane in lanes:
            lane_synergies = champ.get("synergies", {}).get(lane, [])

            for i in range(max_synergies[lane]):
                prefix = f"synergy_{lane}_{i+1}"

                if i < len(lane_synergies):
                    s = lane_synergies[i]
                    row[f"{prefix}_name"] = s.get("champ_counter_i")
                    row[f"{prefix}_winrate"] = s.get("winrate")
                    row[f"{prefix}_games"] = s.get("games")
                    row[f"{prefix}_lane_quality"] = s.get("lane_quality")
                else:
                    row[f"{prefix}_name"] = None
                    row[f"{prefix}_winrate"] = None
                    row[f"{prefix}_games"] = None
                    row[f"{prefix}_lane_quality"] = None

        rows.append(row)

    # ======================
    # 5️⃣ DataFrame + CSV
    # ======================
    df = pd.DataFrame(rows)

    filename = f"{output_path.replace('.csv', '')}_{elo}_{server}_{patch}.csv"
    df.to_csv(filename, index=False)

    print(f"✅ CSV généré : {filename}")
    print(f"📐 Shape DF : {df.shape}")


In [37]:
driver = get_or_create_driver()
driver.get("https://dpm.lol/tierlist?tier=gold_plus")

🔁 Tentative de connexion à Chrome existant...
⏳ Chrome non dispo ou timeout, retry...
🚀 Timeout atteint → lancement d'un nouveau Chrome
🆕 Nouveau Chrome lancé avec debugging


In [14]:
from selenium.webdriver.common.by import By
import time

import sys
sys.stdout.flush()


# Variables pour suivre la combinaison active
elo = None
server = None
patch = None

filters_container_selector = (
    "#root > main > div > div.flex.justify-center.gap-16 > "
    "div.flex.flex-col.max-w-5xl.w-full.items-center.md\\:items-start.my-8.gap-8.px-12.lg\\:px-0 > "
    "div.flex.flex-col.w-full.gap-12.bg-black-800.border.border-black-0\\/10.px-16.py-12.rounded-md > "
    "div.flex.flex-col.lg\\:flex-row.items-center.justify-between.gap-16.lg\\:gap-24.w-full > "
    "div.flex.flex-row.items-center.justify-center.gap-8.lg\\:gap-16"
)
filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)

print("✅ Containers trouvés")

# =========================
# 1️⃣ Ouvrir ELO et noter la liste des boutons
# =========================
filters_container.find_element(By.XPATH, ".//button[1]").click()
time.sleep(0.5)
elo_buttons = get_last_radix_buttons()
print(f"\n🎯 ELO détectés : {[b.text for b in elo_buttons]}")

# =========================
# 2️⃣ Ouvrir SERVER et noter la liste des boutons
# =========================
filters_container.find_element(By.XPATH, ".//button[2]").click()
time.sleep(0.5)
server_buttons = get_last_radix_buttons()
print(f"🌍 SERVER détectés : {[b.text for b in server_buttons]}")

# =========================
# 3️⃣ Ouvrir PATCH et noter la liste des boutons
# =========================
filters_container.find_element(By.XPATH, ".//div/button").click()
time.sleep(0.5)
patch_buttons = get_last_radix_buttons()
print(f"🧩 PATCH détectés : {[b.text for b in patch_buttons]}")

# =========================
# BOUCLE SUR TOUTES LES COMBINAISONS
# =========================


# de elo_départ à elo_max
# for i in range(numeloDepart | 0, max(AeloFin, len(elo_buttons))):
# de elo_départ à elo_fin
# for i in range(numeloDepart | 0, min(AelohFin, len(elo_buttons))): 
for i in range(0,max(1, len(elo_buttons))):
    # 🔁 réouvrir la dropdown ELO
    filters_container.find_element(By.XPATH, ".//button[1]").click()
    time.sleep(0.7)
    elo_buttons = get_last_radix_buttons()
    elo_btn = elo_buttons[i]
    elo = elo_btn.text.strip()


    elo_btn.click()
    print(f"\n🎯 ELO [{i}] cliqué → {elo}")
    time.sleep(1)
    print("1")
    time.sleep(1)
    print("2")

    # de server_départ à server_max
    # for j in range(numServerDepart | 0, max(AServerFin, len(server_buttons))):
    # de server_départ à server_fin
    # for j in range(numServerDepart | 0, min(AServerFin, len(server_buttons))): 
    # for j in range(7, min(8, len(server_buttons))):
    for j in range(0, max(8, len(server_buttons))):
        # if ( ((j >= 3) and (j <= 10) and (j != 4) and(j!=7)) ): #7 pour LAS pour les tests
        if ( ((j >= 3) and (j <= 10) and (j != 4)) ): #7 pour LAS pour les tests
            continue  # 🔹 on skip les serveurs non désirés
        # 🔁 réouvrir la dropdown SERVER
        filters_container.find_element(By.XPATH, ".//button[2]").click()
        time.sleep(0.7)
        server_buttons = get_last_radix_buttons()
        server_btn = server_buttons[j]
        server = server_btn.text.strip()


        server_btn.click()
        print(f"  🌍 SERVER [{j}] cliqué → {server}")
        time.sleep(1)
        print("1")
        time.sleep(1)
        print("2")
        time.sleep(1)
        print("3")
        filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)


        # de patch_départ à patch_max
        # for k in range(numPatchDepart | 0, max(APatchFin, len(patch_buttons))):
        # de patch_départ à patch_fin
        # for k in range(numPatchDepart | 0, min(APatchFin, len(patch_buttons))): 
        for k in range(2, max(5, len(patch_buttons))):  # 🔹 on limite à 3 itérations pour tester          
        # for k in range(5, min(6, len(patch_buttons))):  # 🔹 on limite à 3 itérations pour tester          
            # 🔁 réouvrir la dropdown PATCH
            filters_container.find_element(By.XPATH, ".//div/button").click()
            time.sleep(1)

            # 🔹 récupérer à nouveau les boutons PATCH pour éviter StaleElementReference
            patch_buttons = get_last_radix_buttons()
            time.sleep(1)
            print("click sur les patchs")
            print(f"🧩 PATCH mis à jour : {[b.text for b in patch_buttons]}")
            patch_btn = patch_buttons[k]

            # 🔹 cliquer sur le kème bouton
            patch = patch_btn.text.strip()
            patch_btn.click()
            time.sleep(0.7)

            print(f"    🧩 PATCH [{k}] cliqué → {patch}")

            # ✅ COMBINAISON ACTIVE
            print(f"    ✅ COMBINAISON ACTIVE : ELO={elo}, SERVER={server}, PATCH={patch}")
            time.sleep(1)
            print("1")
            time.sleep(1)
            print("2")
            time.sleep(1)
            print("3")
            
            filters_container = driver.find_element(By.CSS_SELECTOR, filters_container_selector)
            

            champions = init_parse(driver)
            all_champions_data = []  # liste qui va contenir les objets champion
 
            len_champions = len(champions)
            for i in range(len_champions):

            # # for champ in champions[:1]: 
            # for champ in champions: 
                champ = champions[i]
                print(champ)
                champ_data = open_champion_in_new_tab(driver, champ, wait_seconds=5)
                all_champions_data.append(champ_data)

            # for champ in champions[(len(champions)-1):]:  # les 5 derniers pour tester
            #     print(champ)
            #     champ_data = open_champion_in_new_tab(driver, champ, wait_seconds=5)
            #     all_champions_data.append(champ_data)
                if (i % 5 == 0) and (i != 0):  # tous les 5 champions, on sauvegarde dans un CSV intermédiaire pour éviter de tout perdre en cas de bug
                    generate_csv_from_champions(
                        all_champions_data,
                        elo=elo,
                        server=server,
                        patch=patch,
                        output_path=f"{i}_champs_.csv"
                    )    
                    all_champions_data = []
            generate_csv_from_champions(
                all_champions_data,
                elo=elo,
                server=server,
                patch=patch,
                output_path=f"end_champs_.csv"
            )  

            

            driver.execute_script("window.scrollTo(0, arguments[0]);", 0)




✅ Containers trouvés

🎯 ELO détectés : ['Challenger', 'Grandmaster', 'Master+', 'Master', 'Diamond+', 'Diamond', 'Emerald+', 'Emerald', 'Platinum+', 'Platinum', 'Gold+', 'Gold', 'Silver+', 'Bronze', 'Iron', 'TOUT']
🌍 SERVER détectés : ['EUW', 'KR', 'NA', 'BR', 'EUNE', 'JP', 'LAN', 'LAS', 'OCE', 'RU', 'TR', 'VN', 'TOUT']
🧩 PATCH détectés : ['7days', '14days', '30days', '16.3', '16.2', '16.1', '15.24', '15.23', '15.22']

🎯 ELO [0] cliqué → Challenger
1
2
  🌍 SERVER [0] cliqué → EUW
1
2
3
click sur les patchs
🧩 PATCH mis à jour : ['7days', '14days', '30days', '16.3', '16.2', '16.1', '15.24', '15.23', '15.22']
    🧩 PATCH [2] cliqué → 30days
    ✅ COMBINAISON ACTIVE : ELO=Challenger, SERVER=EUW, PATCH=30days
1
2
3
🚀 init_parse() démarré
✅ 185 champions collectés (avant-dernière occurrence si dispo)
{'label': '1', 'numero': 0, 'name': "Vel'Koz", 'scroll_y': 250}
🧭 Scroll vers Y=250 pour 1
🖱️ Ouverture champion → 1
⏳ 1/5s
⏳ 2/5s
⏳ 3/5s
⏳ 1/5s
⏳ 2/5s
⏳ 3/5s
⏳ 4/5s
⏳ 1/5s
⏳ 2/5s
⏳ 3/5s
⏳ 4/5s


KeyboardInterrupt: 

In [24]:
import pandas as pd

pd.set_option("display.max_columns", None)


# chemin vers ton CSV
csv_path = "matchups_champions_Challenger_LAS_16.1.csv"  # adapte si besoin

# ouverture du CSV dans un DataFrame
df = pd.read_csv(csv_path)

# aperçu rapide
df.head()


,champion,role,role_play_ratio,tier,rank,winrate,pickrate,banrate,nb_games_analyzed,matchup_top_1_name,matchup_top_1_winrate,matchup_top_1_games,matchup_top_1_lane_quality,matchup_top_2_name,matchup_top_2_winrate,matchup_top_2_games,matchup_top_2_lane_quality,matchup_top_3_name,matchup_top_3_winrate,matchup_top_3_games,matchup_top_3_lane_quality,matchup_top_4_name,matchup_top_4_winrate,matchup_top_4_games,matchup_top_4_lane_quality,matchup_top_5_name,matchup_top_5_winrate,matchup_top_5_games,matchup_top_5_lane_quality,matchup_top_6_name,matchup_top_6_winrate,matchup_top_6_games,matchup_top_6_lane_quality,matchup_top_7_name,matchup_top_7_winrate,matchup_top_7_games,matchup_top_7_lane_quality,matchup_top_8_name,matchup_top_8_winrate,matchup_top_8_games,matchup_top_8_lane_quality,matchup_top_9_name,matchup_top_9_winrate,matchup_top_9_games,matchup_top_9_lane_quality,matchup_top_10_name,matchup_top_10_winrate,matchup_top_10_games,matchup_top_10_lane_quality,matchup_top_11_name,matchup_top_11_winrate,matchup_top_11_games,matchup_top_11_lane_quality,matchup_top_12_name,matchup_top_12_winrate,matchup_top_12_games,matchup_top_12_lane_quality,matchup_top_13_name,matchup_top_13_winrate,matchup_top_13_games,matchup_top_13_lane_quality,matchup_top_14_name,matchup_top_14_winrate,matchup_top_14_games,matchup_top_14_lane_quality,matchup_top_15_name,matchup_top_15_winrate,matchup_top_15_games,matchup_top_15_lane_quality,matchup_top_16_name,matchup_top_16_winrate,matchup_top_16_games,matchup_top_16_lane_quality,matchup_top_17_name,matchup_top_17_winrate,matchup_top_17_games,matchup_top_17_lane_quality,matchup_top_18_name,matchup_top_18_winrate,matchup_top_18_games,matchup_top_18_lane_quality,matchup_top_19_name,matchup_top_19_winrate,matchup_top_19_games,matchup_top_19_lane_quality,matchup_top_20_name,matchup_top_20_winrate,matchup_top_20_games,matchup_top_20_lane_quality,matchup_top_21_name,matchup_top_21_winrate,matchup_top_21_games,matchup_top_21_lane_quality,matchup_top_22_name,matchup_top_22_winrate,matchup_top_22_games,matchup_top_22_lane_quality,matchup_top_23_name,matchup_top_23_winrate,matchup_top_23_games,matchup_top_23_lane_quality,matchup_top_24_name,matchup_top_24_winrate,matchup_top_24_games,matchup_top_24_lane_quality,matchup_top_25_name,matchup_top_25_winrate,matchup_top_25_games,matchup_top_25_lane_quality,matchup_top_26_name,matchup_top_26_winrate,matchup_top_26_games,matchup_top_26_lane_quality,matchup_top_27_name,matchup_top_27_winrate,matchup_top_27_games,matchup_top_27_lane_quality,matchup_top_28_name,matchup_top_28_winrate,matchup_top_28_games,matchup_top_28_lane_quality,matchup_top_29_name,matchup_top_29_winrate,matchup_top_29_games,matchup_top_29_lane_quality,matchup_top_30_name,matchup_top_30_winrate,matchup_top_30_games,matchup_top_30_lane_quality,matchup_top_31_name,matchup_top_31_winrate,matchup_top_31_games,matchup_top_31_lane_quality,matchup_top_32_name,matchup_top_32_winrate,matchup_top_32_games,matchup_top_32_lane_quality,matchup_top_33_name,matchup_top_33_winrate,matchup_top_33_games,matchup_top_33_lane_quality,matchup_top_34_name,matchup_top_34_winrate,matchup_top_34_games,matchup_top_34_lane_quality,matchup_top_35_name,matchup_top_35_winrate,matchup_top_35_games,matchup_top_35_lane_quality,matchup_top_36_name,matchup_top_36_winrate,matchup_top_36_games,matchup_top_36_lane_quality,matchup_top_37_name,matchup_top_37_winrate,matchup_top_37_games,matchup_top_37_lane_quality,matchup_top_38_name,matchup_top_38_winrate,matchup_top_38_games,matchup_top_38_lane_quality,matchup_top_39_name,matchup_top_39_winrate,matchup_top_39_games,matchup_top_39_lane_quality,matchup_top_40_name,matchup_top_40_winrate,matchup_top_40_games,matchup_top_40_lane_quality,matchup_top_41_name,matchup_top_41_winrate,matchup_top_41_games,matchup_top_41_lane_quality,matchup_top_42_name,matchup_top_42_winrate,matchup_top_42_games,matchup_top_42_lane_quality,matchup_top_43_name,matchup_top_43_winrate,matchup_top_43_games,matchup_top_43_lane_qual

In [21]:
df.columns.tolist()

['champion',
 'role',
 'role_play_ratio',
 'tier',
 'rank',
 'winrate',
 'pickrate',
 'banrate',
 'nb_games_analyzed',
 'matchup_top_1_name',
 'matchup_top_1_winrate',
 'matchup_top_1_games',
 'matchup_top_1_lane_quality',
 'matchup_top_2_name',
 'matchup_top_2_winrate',
 'matchup_top_2_games',
 'matchup_top_2_lane_quality',
 'matchup_top_3_name',
 'matchup_top_3_winrate',
 'matchup_top_3_games',
 'matchup_top_3_lane_quality',
 'matchup_top_4_name',
 'matchup_top_4_winrate',
 'matchup_top_4_games',
 'matchup_top_4_lane_quality',
 'matchup_top_5_name',
 'matchup_top_5_winrate',
 'matchup_top_5_games',
 'matchup_top_5_lane_quality',
 'matchup_top_6_name',
 'matchup_top_6_winrate',
 'matchup_top_6_games',
 'matchup_top_6_lane_quality',
 'matchup_top_7_name',
 'matchup_top_7_winrate',
 'matchup_top_7_games',
 'matchup_top_7_lane_quality',
 'matchup_top_8_name',
 'matchup_top_8_winrate',
 'matchup_top_8_games',
 'matchup_top_8_lane_quality',
 'matchup_top_9_name',
 'matchup_top_9_winrate',


In [22]:
pd.set_option("display.max_colwidth", None)
df.iloc[0]

champion                       Aphelios
role                                adc
role_play_ratio                   99.6%
tier                                 S+
rank                              1 / 2
                                 ...   
synergy_sup_27_lane_quality         NaN
synergy_sup_28_name               Yuumi
synergy_sup_28_winrate            0.00%
synergy_sup_28_games                  1
synergy_sup_28_lane_quality         NaN
Name: 0, Length: 1521, dtype: object